In [1]:
import yaml
import numpy as np
import polars as pl

patho_labels  = ['Pathogenic', 'Likely_pathogenic']
benign_labels = ['Benign', 'Likely_benign']

clinvar_labels = patho_labels + benign_labels

# Create input for VEP and annotation pipeline

In [ ]:
# CLINVAR_URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar_20260621.vcf.gz"

# !wget -P /s/project/ukbbgym/annotation_files/clinvar/ {CLINVAR_URL}


In [ ]:
annotation_dir = "/s/project/ukbbgym/annotation_files/clinvar"
clinvar_vcf_path = "/s/project/ukbbgym/annotation_files/clinvar/clinvar_20260621.vcf.gz"

clinvar = (
    pl.scan_csv(
        clinvar_vcf_path,
        separator="\t",
        comment_prefix="##",
        schema_overrides={"#CHROM": pl.Utf8, "POS": pl.Int64},
        ignore_errors=True,
    )
    .rename({"#CHROM": "chrom", "POS": "pos", "REF": "ref", "ALT": "alt"})
    .with_columns(
        chrom="chr" + pl.col("chrom").cast(pl.Utf8).str.replace(r"^chr", ""),
        clinical_significance=pl.col("INFO").str.extract(r"CLNSIG=([^;]+)", 1),
    )
    .with_columns(
        id=pl.concat_str(["chrom", "pos", "ref", "alt"], separator=":"),
    )
    .select(["id", "chrom", "pos", "ref", "alt", "clinical_significance"])
    .filter(pl.col("clinical_significance").is_in(clinvar_labels))
    .unique(subset="id")          # one row per variant
    .collect()
)

# 1. variant_metadata.parquet — exactly the schema the Snakefile consumes
clinvar.select("id", "chrom", "pos", "ref", "alt").write_parquet(
    f"{annotation_dir}/variant_metadata.parquet"
)

# 2. keep the labels to join back onto the final annotation output on `id`
clinvar.select("id", "clinical_significance").write_parquet(
    f"{annotation_dir}/clinvar_labels.parquet"
)


# Add more missense annotations

In [2]:
import add_missense_variant_annotations as ann

In [3]:
cv_all = (
    pl.read_parquet("/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet")
)

cv_all

id,chrom,pos,ref,alt,region,cds_position,protein_position,distance,amino_acids,gnomade_af,gnomadg_af,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,is_indel,is_insertion,is_deletion,…,gc_is_na,gerpn_is_na,gerps_is_na,grantham_is_na,remapoverlapcl_is_na,remapoverlaptf_is_na,roulette-ar_is_na,roulette-mr_is_na,zoopriphylop_is_na,zoouce_is_na,zooverphylop_is_na,bstatistic_is_na,cdnapos_is_na,dbscsnv-ada_score_is_na,dbscsnv-rf_score_is_na,mamphcons_is_na,mamphylop_is_na,mindisttse_is_na,mindisttss_is_na,mirsvr-aln_is_na,mirsvr-e_is_na,mirsvr-score_is_na,motifdist_is_na,motifecount_is_na,motifehipos_is_na,motifescorechng_is_na,priphcons_is_na,priphylop_is_na,relcdspos_is_na,relprotpos_is_na,relcdnapos_is_na,toverlapmotifs_is_na,targetscan_is_na,verphcons_is_na,verphylop_is_na,cpt1_llr_is_na,blosum62_is_na
str,str,i64,str,str,str,str,str,i64,str,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i8,i8,i8,…,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8
"""chr4:102657480:G:A""","""chr4""",102657480,"""G""","""A""","""ENSG00000109323""",null,null,null,null,null,0.4971,3.57,0.327005,0.0,0,0,0.0,0.0,0.06,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,1.58,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,1,1,1,0,0,0,0,1,1,1,0,1,1,1,0,0,1,1,1,0,1,0,0,1,1
"""chr4:102657672:G:A""","""chr4""",102657672,"""G""","""A""","""ENSG00000109323""",null,null,null,null,0.000351,0.00065,3.607,0.330313,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.003,0.000397,0,0,0,0,0,1,0.24,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,1,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,1,1,1,1,1,0,0,1,1
"""chr4:102657697:G:A""","""chr4""",102657697,"""G""","""A""","""ENSG00000109323""","""1689/2640""","""563/879""",null,"""F""",0.000003,null,8.099,0.77972,0.0,0,0,0.64,0.0,0.01,0.02,0.01,0.005,0.003471,0,0,0,0,0,1,-2.83,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102657758:C:T""","""chr4""",102657758,"""C""","""T""","""ENSG00000109323""","""1628/2640""","""543/879""",null,"""W/*""",6.8420e-7,0.000007,41.0,9.367129,0.0,1,0,0.62,0.0,0.07,0.01,0.0,0.004,0.000411,0,0,0,0,0,1,-9.55,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
"""chr4:102657793:T:C""","""chr4""",102657793,"""T""","""C""","""ENSG00000109323""","""1593/2640""","""531/879""",null,"""V""",0.000003,null,0.495,-0.19568,0.0,0,0,0.6,0.0,0.02,0.02,0.02,0.003,0.000323,0,0,0,0,0,1,-2.03,0.0,0.0,0.0,0,0,0,…,0,0,0,1,0,0,1,0,0,1,0,0,0,1,1,0,0,0,0,1,1,1,1,1,1,1,0,0,0,0,0,1,1,0,0,1,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr17:31352406:AG:A""","""chr17""",31352406,"""AG""","""A""","""ENSG00000196712""","""7608/8520""","""2536/2839""",null,"""K/X""",null,null,0.0,0.28051,0.0,1,0,0.89,0.0,0.01,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,0,1,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"""chr21:45510086:G:GC""","""chr21""",45510086,"""G""","""GC""","""ENSG00000182871""","""3518-3519/4020""","""1173/1339""",null,"""S/SX""",null,null,0.0,0.28051,0.0,1,0,0.88,0.0,0.05,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,1,0,…,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
"""chr17:7224045:GT:G""","""chr17""",7224045,"""GT""","""G""","""ENSG00000072778""","""1411/1968""","""471/655""",null,"""F/X""",null,null,0.0,0.28051,0.0,1,0,0.72,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,0,1,

In [ ]:
ann.main(
    "/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na.parquet",
    fill_null_defaults_path="fill_null_defaults.yaml",
    output_path="/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_missense.parquet",
    download_dir="/s/project/ukbbgym/annotation_files/more_annotations",
    add_alphamissense=False,
    add_cpt1=False,
    add_cadd=False,
    add_gpn_msa=False,
    add_phylop=False,
)

[2026-06-24 17:13:11,140] INFO:add_missense_variant_annotations: === Downloading required files ===
[2026-06-24 17:13:11,143] INFO:add_missense_variant_annotations: $ aria2c -x 16 -s 16 --continue=true --allow-overwrite=true --max-tries=3 --retry-wait=5 --timeout=60 --connect-timeout=30 'https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz' -d '/s/project/ukbbgym/annotation_files/more_missense/tmp' -o 'clinvar.vcf.gz'

06/24 17:13:11 [NOTICE] Downloading 1 item(s)

06/24 17:13:11 [NOTICE] Allocating disk space. Use --file-allocation=none to disable it. See --file-allocation option in man page for more details.
[#591f8e 0B/183MiB(0%) CN:1 DL:0B]
[#591f8e 26MiB/183MiB(14%) CN:8 DL:26MiB ETA:5s]

06/24 17:13:14 [NOTICE] Download complete: /s/project/ukbbgym/annotation_files/more_missense/tmp/clinvar.vcf.gz

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
591f8e|OK  |    77MiB/s|/s/project/ukbb

'/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_missense.parquet'

# Add clinical significance labels to the final annotation output

In [5]:
cv_all = (
    pl.read_parquet("/s/project/ukbbgym/annotation_files/clinvar/vep_annotations_processed_cadd_fill_na_missense.parquet")
)

cv_all

id,chrom,pos,ref,alt,region,cds_position,protein_position,distance,amino_acids,gnomade_af,gnomadg_af,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,is_indel,is_insertion,is_deletion,…,pop_adjusted_esm1v,core_promoter,proximal_promoter,encode_enhancer,encode_promoter,1bp_del,1bp_ins,2_5bp_del,2_5bp_ins,gt_5bp_del,gt_5bp_ins,is_indel_is_na,is_insertion_is_na,is_deletion_is_na,encode_pls_is_na,encode_pels_is_na,encode_dels_is_na,encode_ca_is_na,encode_tf_is_na,not_annotated_in_encode_is_na,revel_score_is_na,clinpred_score_is_na,bayes_del_is_na,mobi_full_disorder_priority_is_na,mobi_curated_disorder_priority_is_na,mobi_full_lip_priority_is_na,ted_domain_is_na,low_complexity_domain_is_na,loftee_hc_is_na,loftee_lc_is_na,plddt,is_pioneer_interface_high,clinical_significance,clinvar_patho,clinvar_likely_patho,clinvar_benign,clinvar_likely_benign
str,str,i64,str,str,str,str,str,i64,str,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i8,i8,i8,…,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,i8,str,i8,i8,i8,i8
"""chr4:102657480:G:A""","""chr4""",102657480,"""G""","""A""","""ENSG00000109323""",null,null,null,null,null,0.4971,3.57,0.327005,0.0,0,0,0.0,0.0,0.06,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,1.58,0.0,0.0,0.0,0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,null,0,"""Benign""",0,0,1,0
"""chr4:102657672:G:A""","""chr4""",102657672,"""G""","""A""","""ENSG00000109323""",null,null,null,null,0.000351,0.00065,3.607,0.330313,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.003,0.000397,0,0,0,0,0,1,0.24,0.0,0.0,0.0,0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,null,0,"""Benign""",0,0,1,0
"""chr4:102657697:G:A""","""chr4""",102657697,"""G""","""A""","""ENSG00000109323""","""1689/2640""","""563/879""",null,"""F""",0.000003,null,8.099,0.77972,0.0,0,0,0.64,0.0,0.01,0.02,0.01,0.005,0.003471,0,0,0,0,0,1,-2.83,0.0,0.0,0.0,0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,98.690002,0,"""Likely_benign""",0,0,0,1
"""chr4:102657758:C:T""","""chr4""",102657758,"""C""","""T""","""ENSG00000109323""","""1628/2640""","""543/879""",null,"""W/*""",6.8420e-7,0.000007,41.0,9.367129,0.0,1,0,0.62,0.0,0.07,0.01,0.0,0.004,0.000411,0,0,0,0,0,1,-9.55,0.0,0.0,0.0,0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,98.059998,0,"""Pathogenic""",1,0,0,0
"""chr4:102657793:T:C""","""chr4""",102657793,"""T""","""C""","""ENSG00000109323""","""1593/2640""","""531/879""",null,"""V""",0.000003,null,0.495,-0.19568,0.0,0,0,0.6,0.0,0.02,0.02,0.02,0.003,0.000323,0,0,0,0,0,1,-2.03,0.0,0.0,0.0,0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,98.809998,0,"""Likely_benign""",0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr17:31352406:AG:A""","""chr17""",31352406,"""AG""","""A""","""ENSG00000196712""","""7608/8520""","""2536/2839""",null,"""K/X""",null,null,0.0,0.28051,0.0,1,0,0.89,0.0,0.01,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,0,1,…,null,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,null,0,"""Pathogenic""",1,0,0,0
"""chr21:45510086:G:GC""","""chr21""",45510086,"""G""","""GC""","""ENSG00000182871""","""3518-3519/4020""","""1173/1339""",null,"""S/SX""",null,null,0.0,0.28051,0.0,1,0,0.88,0.0,0.05,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,1,0,…,null,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,47.779999,0,"""Pathogenic""",1,0,0,0
"""chr17:7224045:G

In [9]:
cv_all.columns

['id',
 'chrom',
 'pos',
 'ref',
 'alt',
 'region',
 'cds_position',
 'protein_position',
 'distance',
 'amino_acids',
 'gnomade_af',
 'gnomadg_af',
 'cadd_phred',
 'cadd_raw',
 'am_pathogenicity',
 'loftee_hc',
 'loftee_lc',
 'relative_cds_position',
 'next_in_frame_relative',
 'spliceai_delta_score',
 'pangolin_score',
 'delta_score',
 'absplice_dna_max',
 'absplice2_max',
 'five_prime_utr_variant_consequence_uaug_gained',
 'five_prime_utr_variant_consequence_uaug_lost',
 'five_prime_utr_variant_consequence_uframeshift',
 'five_prime_utr_variant_consequence_ustop_gained',
 'five_prime_utr_variant_consequence_ustop_lost',
 'variant_length',
 'gpn_score',
 'promoterai',
 'score_pai3d',
 'blosum62',
 'is_indel',
 'is_insertion',
 'is_deletion',
 'mobi_full_disorder_priority',
 'mobi_curated_disorder_priority',
 'mobi_full_lip_priority',
 'ted_domain',
 'low_complexity_domain',
 'cpt1_llr',
 'encode_pls',
 'encode_pels',
 'encode_dels',
 'encode_ca',
 'encode_ca-ctcf',
 'encode_ca-tf',
 

In [6]:
annotation_dir = "/s/project/ukbbgym/annotation_files/clinvar"
cv_labels = pl.read_parquet(f"{annotation_dir}/clinvar_labels_20260621.parquet")
cv_labels

id,clinical_significance
str,str
"""chr6:161360200:G:A""","""Likely_benign"""
"""chr21:44287497:C:T""","""Likely_benign"""
"""chr17:67911214:A:G""","""Likely_benign"""
"""chr17:1676275:G:A""","""Likely_benign"""
"""chr11:6640711:A:G""","""Likely_benign"""
…,…
"""chr19:39499413:C:G""","""Likely_benign"""
"""chr19:17787812:C:T""","""Likely_benign"""
"""chr3:184355543:G:C""","""Likely_benign"""


In [7]:
cv_all_labs = (
    cv_labels
    .join(
        cv_all,
        on="id",
        # how="left",
        validate="1:m"
    )
)

cv_all_labs

id,clinical_significance,chrom,pos,ref,alt,region,cds_position,protein_position,distance,amino_acids,gnomade_af,gnomadg_af,cadd_phred,cadd_raw,am_pathogenicity,loftee_hc,loftee_lc,relative_cds_position,next_in_frame_relative,spliceai_delta_score,pangolin_score,delta_score,absplice_dna_max,absplice2_max,five_prime_utr_variant_consequence_uaug_gained,five_prime_utr_variant_consequence_uaug_lost,five_prime_utr_variant_consequence_uframeshift,five_prime_utr_variant_consequence_ustop_gained,five_prime_utr_variant_consequence_ustop_lost,variant_length,gpn_score,promoterai,score_pai3d,blosum62,is_indel,is_insertion,…,pop_adjusted_esm1v,core_promoter,proximal_promoter,encode_enhancer,encode_promoter,1bp_del,1bp_ins,2_5bp_del,2_5bp_ins,gt_5bp_del,gt_5bp_ins,is_indel_is_na,is_insertion_is_na,is_deletion_is_na,encode_pls_is_na,encode_pels_is_na,encode_dels_is_na,encode_ca_is_na,encode_tf_is_na,not_annotated_in_encode_is_na,revel_score_is_na,clinpred_score_is_na,bayes_del_is_na,mobi_full_disorder_priority_is_na,mobi_curated_disorder_priority_is_na,mobi_full_lip_priority_is_na,ted_domain_is_na,low_complexity_domain_is_na,loftee_hc_is_na,loftee_lc_is_na,plddt,is_pioneer_interface_high,clinical_significance_right,clinvar_patho,clinvar_likely_patho,clinvar_benign,clinvar_likely_benign
str,str,str,i64,str,str,str,str,str,i64,str,f32,f32,f32,f32,f32,i8,i8,f32,f32,f32,f32,f32,f32,f32,u8,u8,u8,u8,u8,u32,f32,f32,f32,f32,i8,i8,…,f32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,i8,str,i8,i8,i8,i8
"""chr4:102657480:G:A""","""Benign""","""chr4""",102657480,"""G""","""A""","""ENSG00000109323""",null,null,null,null,null,0.4971,3.57,0.327005,0.0,0,0,0.0,0.0,0.06,0.0,0.0,0.001,0.000033,0,0,0,0,0,1,1.58,0.0,0.0,0.0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,null,0,"""Benign""",0,0,1,0
"""chr4:102657672:G:A""","""Benign""","""chr4""",102657672,"""G""","""A""","""ENSG00000109323""",null,null,null,null,0.000351,0.00065,3.607,0.330313,0.0,0,0,0.0,0.0,0.0,0.0,0.0,0.003,0.000397,0,0,0,0,0,1,0.24,0.0,0.0,0.0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,null,0,"""Benign""",0,0,1,0
"""chr4:102657697:G:A""","""Likely_benign""","""chr4""",102657697,"""G""","""A""","""ENSG00000109323""","""1689/2640""","""563/879""",null,"""F""",0.000003,null,8.099,0.77972,0.0,0,0,0.64,0.0,0.01,0.02,0.01,0.005,0.003471,0,0,0,0,0,1,-2.83,0.0,0.0,0.0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,98.690002,0,"""Likely_benign""",0,0,0,1
"""chr4:102657758:C:T""","""Pathogenic""","""chr4""",102657758,"""C""","""T""","""ENSG00000109323""","""1628/2640""","""543/879""",null,"""W/*""",6.8420e-7,0.000007,41.0,9.367129,0.0,1,0,0.62,0.0,0.07,0.01,0.0,0.004,0.000411,0,0,0,0,0,1,-9.55,0.0,0.0,0.0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,98.059998,0,"""Pathogenic""",1,0,0,0
"""chr4:102657793:T:C""","""Likely_benign""","""chr4""",102657793,"""T""","""C""","""ENSG00000109323""","""1593/2640""","""531/879""",null,"""V""",0.000003,null,0.495,-0.19568,0.0,0,0,0.6,0.0,0.02,0.02,0.02,0.003,0.000323,0,0,0,0,0,1,-2.03,0.0,0.0,0.0,0,0,…,null,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,98.809998,0,"""Likely_benign""",0,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr17:31352406:AG:A""","""Pathogenic""","""chr17""",31352406,"""AG""","""A""","""ENSG00000196712""","""7608/8520""","""2536/2839""",null,"""K/X""",null,null,0.0,0.28051,0.0,1,0,0.89,0.0,0.01,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,0,…,null,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,null,0,"""Pathogenic""",1,0,0,0
"""chr21:45510086:G:GC""","""Pathogenic""","""chr21""",45510086,"""G""","""GC""","""ENSG00000182871""","""3518-3519/4020""","""1173/1339""",null,"""S/SX""",null,null,0.0,0.28051,0.0,1,0,0.88,0.0,0.05,0.0,0.0,0.0,0.0,0,0,0,0,0,2,-1.41,0.0,0.0,0.0,1,1,

In [8]:
cv_all_labs.write_parquet(
    f"{annotation_dir}/clinvar_significance_vep_annotations_processed_cadd_fill_na_20260621.parquet"
)